In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score


# Load dataset

df = pd.read_csv("Dataset1.csv")

df = df.sort_values(["Train_No", "SN"])


# Convert time

df["Arrival_time"] = pd.to_datetime(
    df["Arrival_time"],
    format="%H:%M:%S"
)

df["Departure_Time"] = pd.to_datetime(
    df["Departure_Time"],
    format="%H:%M:%S"
)


# Prepare data

model_data = df.groupby("Train_No").agg(
    Total_Distance=("Distance","max"),
    Number_of_Stops=("Station_Name","count"),
    Start_Time=("Departure_Time","first"),
    End_Time=("Arrival_time","last")
).reset_index()


# Handle next day journeys

model_data.loc[
    model_data["End_Time"] < model_data["Start_Time"],
    "End_Time"
] += pd.Timedelta(days=1)


# Journey hours

model_data["Journey_Hours"] = (
    model_data["End_Time"]
    -
    model_data["Start_Time"]
).dt.total_seconds()/3600



# Features and target

X = model_data[
    [
        "Total_Distance",
        "Number_of_Stops"
    ]
]

y = model_data["Journey_Hours"]



# Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)



# Train model

model = LinearRegression()

model.fit(X_train, y_train)



# Predict

y_pred = model.predict(X_test)



# Evaluation

evaluation = pd.DataFrame({
    "Metric": [
        "Mean Absolute Error",
        "Mean Squared Error",
        "R2 Score"
    ],
    
    "Value": [
        mean_absolute_error(y_test,y_pred),
        mean_squared_error(y_test,y_pred),
        r2_score(y_test,y_pred)
    ]
})


evaluation

,Metric,Value
0,Mean Absolute Error,2.536869
1,Mean Squared Error,15.653459
2,R2 Score,0.450737
